[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/075.%20The%20VRP%20with%20Pickup%20and%20Delivery%20%28VRPPD%29/P75-Tier-1_executed.ipynb)

---

# 75. The VRP with Pickup and Delivery (VRPPD)
## Tier 1: The Pen-and-Paper Method (Mathematical Formulation)

A vehicle routing problem with pickup and delivery (VRPPD) assigns vehicle routes so that every pickup and its corresponding delivery are served. Each pair must stay on the same vehicle, the pickup must occur before the delivery, and the vehicle's capacity must not be exceeded. This notebook formulates the problem as a mixed-integer linear program (MILP) and solves it with PuLP's CBC solver.

### Purpose, inputs, and outputs

The example has three pickup-delivery pairs and two vehicles. Its inputs are a directed distance-cost matrix, signed customer demands, the fleet size, and a common vehicle capacity. The notebook reports the solver status, routes, route distances, load profiles, a route visualization, and a capacity sensitivity analysis. Distances are used both as travel costs in the objective and as modeled travel durations in the route-order constraints; the example treats both as the same abstract units.

A directed distance-cost matrix means that the cost from vertex $i$ to vertex $j$ is the matrix entry $d_{ij}$. In general, $d_{ij}$ need not equal $d_{ji}$. A signed demand is positive for a pickup, which adds goods to the vehicle, and negative for a delivery, which removes goods.

### Scope and modeling assumptions

- Each customer is visited exactly once, and each pickup and its paired delivery are served by the same vehicle.
- Demand is positive at a pickup and negative at its paired delivery. Each pair has equal pickup and delivery quantities, so a vehicle returns with zero physical load after completing its assigned pairs.
- All vehicles leave the start depot once and return to the end depot once. A vehicle can remain unused by traveling directly between the two depot nodes.
- Every vehicle has the same capacity. Time windows, service durations, vehicle-specific costs, and changes during a route are outside this model.
- The distance matrix defines routing costs and modeled travel durations. The plotted node coordinates are only a schematic layout and do not determine those values.
- The time-based argument that excludes disconnected customer cycles requires strictly positive travel durations between every two distinct customer vertices. The instance checks this assumption before building the model.

### Solution sequence

1. Define the pickup, delivery, vehicle, and vertex sets and the instance data.
2. Create binary routing variables and continuous load and time variables.
3. Minimize total travel distance while enforcing visitation, route continuity, pairing, capacity, and precedence.
4. Solve the MILP with CBC, extract routes only when the solver reports optimality, and check the reported routes.
5. Visualize the solution and solve the same model for a range of vehicle capacities.

A mixed-integer linear program combines continuous variables with discrete choices, such as whether an arc is used. CBC may find a feasible route without proving that no shorter route exists. Therefore, only the **Optimal** status is treated as an optimality certificate; other statuses are reported without claiming infeasibility or optimality.

In [ ]:
from dataclasses import dataclass, field
from math import isclose
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pulp

print("Libraries imported successfully for the VRPPD mathematical formulation.")

In [ ]:
@dataclass
class VRPPDInstance:
    """Data and derived vertex sets for a pickup-and-delivery instance."""

    num_pairs: int
    num_vehicles: int
    vehicle_capacity: int
    distances: np.ndarray
    demands: List[int]
    num_vertices: int = field(init=False)
    depot_start: int = field(init=False)
    depot_end: int = field(init=False)
    pickups: List[int] = field(init=False)
    deliveries: List[int] = field(init=False)
    all_vertices: List[int] = field(init=False)
    vehicles: List[int] = field(init=False)

    def __post_init__(self):
        if self.num_pairs <= 0 or self.num_vehicles <= 0:
            raise ValueError("The instance must have at least one pair and one vehicle.")
        if self.vehicle_capacity <= 0:
            raise ValueError("Vehicle capacity must be positive.")

        self.num_vertices = 2 * self.num_pairs + 2
        if self.distances.shape != (self.num_vertices, self.num_vertices):
            raise ValueError("The distance matrix must have one row and column per vertex.")
        if not np.all(np.isfinite(self.distances)):
            raise ValueError("Travel distances must be finite numbers.")
        if np.any(self.distances < 0):
            raise ValueError("Travel distances must be nonnegative.")
        if len(self.demands) != 2 * self.num_pairs:
            raise ValueError("There must be one signed demand for each customer vertex.")

        self.depot_start = 0
        self.depot_end = 2 * self.num_pairs + 1
        self.pickups = list(range(1, self.num_pairs + 1))
        self.deliveries = list(range(self.num_pairs + 1, 2 * self.num_pairs + 1))
        self.all_vertices = list(range(self.num_vertices))
        self.vehicles = list(range(self.num_vehicles))

        customer_distances = self.distances[1:self.depot_end, 1:self.depot_end]
        distinct_customer_arcs = ~np.eye(2 * self.num_pairs, dtype=bool)
        if np.any(customer_distances[distinct_customer_arcs] <= 0):
            raise ValueError(
                "Travel durations between distinct customer vertices must be positive."
            )

        for pickup in self.pickups:
            delivery = pickup + self.num_pairs
            if self.demands[pickup - 1] <= 0 or self.demands[delivery - 1] >= 0:
                raise ValueError("Pickup demands must be positive and delivery demands negative.")
            if self.demands[pickup - 1] + self.demands[delivery - 1] != 0:
                raise ValueError("Each delivery must balance its paired pickup demand.")


def create_concrete_instance():
    """Create the three-pair, two-vehicle example."""
    num_pairs = 3
    num_vehicles = 2
    vehicle_capacity = 6

    # Positive values are pickups; negative values are paired deliveries.
    demands = [3, 2, 4, -3, -2, -4]

    # Vertex order: start depot, P1-P3, D1-D3, end depot.
    distances = np.array([
        [0, 8, 6, 7, 9, 8, 12, 0],
        [8, 0, 4, 6, 5, 6, 9, 8],
        [6, 4, 0, 3, 7, 5, 8, 6],
        [7, 6, 3, 0, 9, 7, 6, 7],
        [9, 5, 7, 9, 0, 3, 4, 9],
        [8, 6, 5, 7, 3, 0, 2, 8],
        [12, 9, 8, 6, 4, 2, 0, 12],
        [0, 8, 6, 7, 9, 8, 12, 0],
    ])

    return VRPPDInstance(
        num_pairs=num_pairs,
        num_vehicles=num_vehicles,
        vehicle_capacity=vehicle_capacity,
        distances=distances,
        demands=demands,
    )


instance = create_concrete_instance()
print("VRPPD instance created:")
print(f"- Pickup-delivery pairs: {instance.num_pairs}")
print(f"- Vehicles: {instance.num_vehicles}")
print(f"- Vehicle capacity: {instance.vehicle_capacity}")
print(f"- Total vertices, including both depots: {instance.num_vertices}")
print(f"- Signed customer demands: {instance.demands}")

## MILP formulation

The model chooses whether each vehicle travels along each possible directed arc. An arc from $i$ to $j$ means that the vehicle goes directly from vertex $i$ to vertex $j$; direction matters because the cost matrix is defined for ordered pairs. Binary routing decisions describe the route. Continuous load and time variables are auxiliary states that let linear constraints enforce capacity and route order without changing the objective.

### Sets, data, and decision variables

Let $P=\{1,\ldots,n\}$ be the pickup vertices, $D=\{n+1,\ldots,2n\}$ the delivery vertices, $K=\{0,\ldots,m-1\}$ the vehicles, and $V=\{0,\ldots,2n+1\}$ all vertices. Vertex $0$ is the start depot and vertex $2n+1$ is the end depot. For pickup $i\in P$, its paired delivery is $i+n$. The code represents these sets by `instance.pickups`, `instance.deliveries`, `instance.vehicles`, and `instance.all_vertices`.

For distinct vertices $i,j\in V$, let $d_{ij}$ be the travel cost and modeled travel duration from $i$ to $j$. Let $q_i$ be the signed demand at customer $i$, and let $Q$ be the common vehicle capacity. Depot demands are zero. In the code, these data are `instance.distances[i, j]`, `node_demand(i)`, and `instance.vehicle_capacity`.

The binary variable $x_{ijk}$ is 1 when vehicle $k$ travels directly from $i$ to $j$, and 0 otherwise. The continuous variable $u_{ik}$ is a load-state auxiliary variable after vertex $i$. Its constraints make it a capacity-enforcing lower bound on cumulative signed demand at customer vertices; depot load states are fixed to zero. It is not necessarily the exact physical load at every vertex. The continuous variable $s_{ik}$ is the cumulative modeled travel time at vertex $i$ for vehicle $k$. The code stores these variables as `x[i, j, k]`, `u[i, k]`, and `s[i, k]`.

### Objective: minimize total travel distance

The objective adds the cost of every selected arc:

$$\min \quad \sum_{k\in K}\sum_{i\in V}\sum_{\substack{j\in V\\j\ne i}} d_{ij}x_{ijk}.$$

Here, $x_{ijk}$ determines whether an arc contributes to the total, and $d_{ij}$ is that arc's cost. The corresponding code uses `pulp.lpSum` over all distinct vertex pairs and vehicles.

### Visit, connect, and direct each route

Every customer has exactly one outgoing selected arc across the fleet. Flow conservation requires each vehicle that enters a customer to leave it. Each vehicle has one departure from the start depot and one arrival at the end depot:

$$\sum_{k\in K}\sum_{\substack{j\in V\\j\ne i}}x_{ijk}=1 \qquad i\in P\cup D,$$

$$\sum_{\substack{j\in V\\j\ne 0}}x_{0jk}=1, \qquad
\sum_{\substack{i\in V\\i\ne 2n+1}}x_{i,2n+1,k}=1 \qquad k\in K,$$

$$\sum_{\substack{j\in V\\j\ne i}}x_{ijk}
=\sum_{\substack{j\in V\\j\ne i}}x_{jik}
\qquad i\in P\cup D,\ k\in K.$$

Arcs entering the start depot and arcs leaving the end depot are prohibited. Together, the visit and flow constraints ensure each customer has one incoming and one outgoing selected arc, assigned consistently to a vehicle. The depot constraints require every vehicle to have a route; a vehicle with no customers can use the direct start-to-end arc.

### Pairing and precedence

For each pair, the pickup and delivery must be assigned to the same vehicle. The delivery must occur after its pickup:

$$\sum_{\substack{j\in V\\j\ne i}}x_{ijk}
=\sum_{\substack{j\in V\\j\ne i+n}}x_{i+n,j,k}
\qquad i\in P,\ k\in K,$$

$$s_{i+n,k}\ge s_{ik}\qquad i\in P,\ k\in K.$$

The first equation equates the assignment of both locations to each vehicle. The second records the precedence requirement for that vehicle. Since the pickup and delivery share a vehicle, cumulative time increases along the selected route, and distinct customer-to-customer durations are positive, this inequality is consistent with the visit order enforced by the time transitions.

### Capacity and route-order transitions

For a selected arc ending at a customer, the load state at that customer must account for its signed demand. For every selected arc that can advance a route, modeled time must increase by the arc duration. Big-$M$ terms deactivate these relationships when an arc is not selected:

$$u_{jk}\ge u_{ik}+q_j-M_L(1-x_{ijk})
\qquad i\in V,\ j\in P\cup D,\ i\ne j,\ k\in K,$$

$$s_{jk}\ge s_{ik}+d_{ij}-M_T(1-x_{ijk})
\qquad i\in V\setminus\{2n+1\},\ j\in V\setminus\{0\},\ i\ne j,\ k\in K.$$

The variable domains and depot conditions are:

$$x_{ijk}\in\{0,1\},\qquad u_{ik}\in[0,Q],\qquad s_{ik}\in[0,H].$$

Arcs entering the start depot and arcs leaving the end depot are fixed to zero. The load state is zero at both depots, and time is zero at the start depot. A load transition is applied when the destination is a customer; it is not applied on arrival at the end depot, whose load-state variable is fixed to zero. The capacity of the physical load arriving there has already been enforced at the final customer. The code uses

$$M_L=Q+\max_{i\in P\cup D}|q_i|,\qquad
H=(2n+1)\max_{i,j\in V}d_{ij},\qquad
M_T=H+\max_{i,j\in V}d_{ij}.$$

The bound $M_L$ is sufficient to deactivate a load transition on an unselected arc: the largest possible $u_{ik}+q_j$ is at most $Q+\max_i|q_i|$. The horizon $H$ bounds a route that visits at most $2n$ customers and uses at most $2n+1$ arcs. The value $M_T$ is large enough that an unselected arc imposes no additional time restriction within this horizon. These quantities are computed in `solve_vrppd_mip` as `big_m_load`, `route_horizon`, and `big_m_time`.

Along a selected route, the load transition makes $u_{ik}$ at least the cumulative signed demand after visiting customer vertex $i$. Because $u_{ik}$ is nonnegative and at most $Q$, any route whose physical cumulative load exceeds $Q$ is infeasible. Pair precedence ensures that a delivery cannot reduce load before its corresponding pickup. The load state is an auxiliary capacity certificate, whereas the load profile later reported for a route is calculated directly from the signed demands.

Flow conservation alone could allow disconnected customer cycles. On every selected customer-to-customer arc, time must increase by the strictly positive duration. A disconnected cycle would therefore require time to increase all the way around a loop and return to its starting value, which is impossible. This is why positivity of those durations is a model assumption checked when the instance is created.

### Model size and implementation

Here, $n$ is the number of pickup-delivery pairs, $m$ is the number of vehicles, and $|V|=2n+2$. The model has $O(m|V|^2)$ binary routing variables and a comparable number of arc-related constraints. This compact MILP formulation can become computationally demanding as the instance grows because the solver must search among discrete routing choices.

The function `solve_vrppd_mip` creates the variables, objective, and constraints, then calls CBC. Routing variables are binary, load variables are bounded by vehicle capacity, and time variables are nonnegative with their horizon enforced by model constraints.

In [ ]:
def solve_vrppd_mip(instance: VRPPDInstance):
    """Build and solve the VRPPD mixed-integer linear program."""
    prob = pulp.LpProblem("VRPPD", pulp.LpMinimize)

    x = {}
    for i in instance.all_vertices:
        for j in instance.all_vertices:
            if i != j:
                for k in instance.vehicles:
                    x[i, j, k] = pulp.LpVariable(
                        f"x_{i}_{j}_{k}", 0, 1, pulp.LpBinary
                    )

    u = {}
    s = {}
    for i in instance.all_vertices:
        for k in instance.vehicles:
            u[i, k] = pulp.LpVariable(
                f"u_{i}_{k}", 0, instance.vehicle_capacity
            )
            s[i, k] = pulp.LpVariable(f"s_{i}_{k}", 0)

    def node_demand(vertex):
        if vertex in instance.pickups + instance.deliveries:
            return instance.demands[vertex - 1]
        return 0

    prob += pulp.lpSum(
        instance.distances[i, j] * x[i, j, k]
        for i in instance.all_vertices
        for j in instance.all_vertices
        for k in instance.vehicles
        if i != j
    ), "Total_travel_distance"

    # Depot direction restrictions ensure every route runs from start to end.
    for k in instance.vehicles:
        for i in instance.all_vertices:
            if i != instance.depot_start:
                prob += x[i, instance.depot_start, k] == 0, f"No_arc_into_start_{i}_{k}"
            if i != instance.depot_end:
                prob += x[instance.depot_end, i, k] == 0, f"No_arc_out_of_end_{i}_{k}"

    # Each customer has one outgoing arc across the fleet.
    for i in instance.pickups + instance.deliveries:
        prob += pulp.lpSum(
            x[i, j, k]
            for j in instance.all_vertices
            for k in instance.vehicles
            if i != j
        ) == 1, f"Visit_customer_{i}"

    # Every vehicle has one start-depot departure and one end-depot arrival.
    for k in instance.vehicles:
        prob += pulp.lpSum(
            x[instance.depot_start, j, k]
            for j in instance.all_vertices
            if j != instance.depot_start
        ) == 1, f"Leave_start_depot_{k}"
        prob += pulp.lpSum(
            x[i, instance.depot_end, k]
            for i in instance.all_vertices
            if i != instance.depot_end
        ) == 1, f"Return_to_end_depot_{k}"

    # Flow conservation links each vehicle's arrival and departure at a customer.
    for i in instance.pickups + instance.deliveries:
        for k in instance.vehicles:
            prob += pulp.lpSum(
                x[i, j, k] for j in instance.all_vertices if i != j
            ) == pulp.lpSum(
                x[j, i, k] for j in instance.all_vertices if i != j
            ), f"Flow_{i}_{k}"

    # Equal vehicle assignments keep every pickup with its paired delivery.
    for pickup in instance.pickups:
        delivery = pickup + instance.num_pairs
        for k in instance.vehicles:
            prob += pulp.lpSum(
                x[pickup, j, k]
                for j in instance.all_vertices
                if pickup != j
            ) == pulp.lpSum(
                x[delivery, j, k]
                for j in instance.all_vertices
                if delivery != j
            ), f"Pair_on_same_vehicle_{pickup}_{k}"

    max_abs_demand = max(abs(demand) for demand in instance.demands)
    big_m_load = instance.vehicle_capacity + max_abs_demand
    max_distance = float(np.max(instance.distances))
    route_horizon = (2 * instance.num_pairs + 1) * max_distance
    big_m_time = route_horizon + max_distance

    for k in instance.vehicles:
        prob += u[instance.depot_start, k] == 0, f"Initial_load_{k}"
        prob += u[instance.depot_end, k] == 0, f"Final_load_{k}"
        prob += s[instance.depot_start, k] == 0, f"Start_time_{k}"
        for i in instance.all_vertices:
            prob += s[i, k] <= route_horizon, f"Time_horizon_{i}_{k}"

    # Customer load transitions enforce capacity; time transitions establish route order.
    for i in instance.all_vertices:
        for j in instance.all_vertices:
            if i == j or j == instance.depot_start or i == instance.depot_end:
                continue
            for k in instance.vehicles:
                if j != instance.depot_end:
                    prob += u[j, k] >= (
                        u[i, k]
                        + node_demand(j)
                        - big_m_load * (1 - x[i, j, k])
                    ), f"Load_transition_{i}_{j}_{k}"
                prob += s[j, k] >= (
                    s[i, k]
                    + instance.distances[i, j]
                    - big_m_time * (1 - x[i, j, k])
                ), f"Time_transition_{i}_{j}_{k}"

    # This explicit condition records the precedence requirement for each pair.
    for pickup in instance.pickups:
        delivery = pickup + instance.num_pairs
        for k in instance.vehicles:
            prob += s[delivery, k] >= s[pickup, k], f"Pickup_before_delivery_{pickup}_{k}"

    print("Solving the VRPPD MILP with CBC...")
    prob.solve(pulp.PULP_CBC_CMD(msg=False, timeLimit=30))
    return prob, x, u, s


prob, x, u, s = solve_vrppd_mip(instance)
solution_status = pulp.LpStatus[prob.status]
print(f"\nSolution status: {solution_status}")

routes = []
if solution_status == "Optimal":
    print(f"Objective value (total distance): {pulp.value(prob.objective):.2f}")
else:
    print("An optimal solution was not certified within the solver limit.")

In [ ]:
def extract_solution(instance: VRPPDInstance, x, u, s):
    """Extract routes and load profiles from an optimal MILP solution."""
    routes = []
    visited_customers = []

    for k in instance.vehicles:
        route = [instance.depot_start]
        current = instance.depot_start
        load_profile = [0]
        seen_on_route = {instance.depot_start}

        while current != instance.depot_end:
            next_vertices = [
                j
                for j in instance.all_vertices
                if j != current
                and (current, j, k) in x
                and pulp.value(x[current, j, k]) is not None
                and pulp.value(x[current, j, k]) > 0.5
            ]
            if len(next_vertices) != 1:
                raise RuntimeError(
                    f"Vehicle {k + 1} must have exactly one selected successor at vertex {current}."
                )

            next_vertex = next_vertices[0]
            if next_vertex in seen_on_route:
                raise RuntimeError(f"Vehicle {k + 1} route contains a cycle.")

            route.append(next_vertex)
            seen_on_route.add(next_vertex)
            if next_vertex in instance.pickups + instance.deliveries:
                load_profile.append(
                    load_profile[-1] + instance.demands[next_vertex - 1]
                )
                visited_customers.append(next_vertex)
            else:
                load_profile.append(load_profile[-1])
            current = next_vertex

        if not isclose(load_profile[-1], 0.0, abs_tol=1e-9):
            raise AssertionError("A completed route must return to the depot with zero load.")
        if max(load_profile) > instance.vehicle_capacity:
            raise AssertionError("An extracted route exceeds vehicle capacity.")
        if min(load_profile) < 0:
            raise AssertionError("An extracted route has a negative vehicle load.")

        route_distance = sum(
            instance.distances[route[index], route[index + 1]]
            for index in range(len(route) - 1)
        )
        routes.append({
            "vehicle": k + 1,
            "route": route,
            "load": load_profile[-1],
            "load_profile": load_profile,
            "peak_load": max(load_profile),
            "distance": float(route_distance),
        })

    expected_customers = instance.pickups + instance.deliveries
    if sorted(visited_customers) != expected_customers:
        raise AssertionError("The extracted routes must visit every customer exactly once.")

    for pickup in instance.pickups:
        delivery = pickup + instance.num_pairs
        serving_routes = [
            route_info["route"]
            for route_info in routes
            if pickup in route_info["route"]
        ]
        if len(serving_routes) != 1 or delivery not in serving_routes[0]:
            raise AssertionError("Each pickup and delivery must be served by the same vehicle.")
        if serving_routes[0].index(pickup) >= serving_routes[0].index(delivery):
            raise AssertionError("Each pickup must precede its paired delivery.")

    return routes


if solution_status == "Optimal":
    routes = extract_solution(instance, x, u, s)
    route_distance_total = sum(route_info["distance"] for route_info in routes)
    if not isclose(
        route_distance_total,
        pulp.value(prob.objective),
        rel_tol=1e-9,
        abs_tol=1e-9,
    ):
        raise AssertionError("The extracted route distances must match the MILP objective.")

    print("\n=== OPTIMAL SOLUTION ===")
    print(f"Total distance: {pulp.value(prob.objective):.2f}")
    print("\nVehicle routes:")
    for route_info in routes:
        print(f"\nVehicle {route_info['vehicle']}:")
        print(f"  Route (vertex indices): {' → '.join(map(str, route_info['route']))}")
        print(f"  Ending load: {route_info['load']:.0f}")
        print(f"  Peak load: {route_info['peak_load']:.0f} of {instance.vehicle_capacity}")
        print(f"  Distance: {route_info['distance']:.2f}")
        print("  Pickup-delivery precedence: satisfied")

In [ ]:
def visualize_solution(instance: VRPPDInstance, routes):
    """Plot route structure, load profiles, distances, and solution metrics."""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle("VRPPD MILP Solution Analysis", fontsize=16, fontweight="bold")

    positions = {
        0: (0, 0),
        1: (2, 3),
        2: (4, 2),
        3: (6, 4),
        4: (3, 1),
        5: (5, 0),
        6: (7, 2),
        7: (0, 0),
    }
    colors = ["tab:blue", "tab:red", "tab:green", "tab:orange"]

    ax1.set_title("Schematic Route Network", fontweight="bold")
    for index, route_info in enumerate(routes):
        route = route_info["route"]
        color = colors[index % len(colors)]
        for position in range(len(route) - 1):
            start_position = positions[route[position]]
            end_position = positions[route[position + 1]]
            ax1.annotate(
                "",
                xy=end_position,
                xytext=start_position,
                arrowprops={"arrowstyle": "->", "color": color, "lw": 2, "alpha": 0.7},
            )
        ax1.plot([], [], color=color, linewidth=2, label=f"Vehicle {route_info['vehicle']}")

    for vertex, position in positions.items():
        if vertex in (instance.depot_start, instance.depot_end):
            if vertex == instance.depot_start:
                ax1.plot(position[0], position[1], "ks", markersize=12)
                ax1.text(position[0], position[1] + 0.3, "Depot", ha="center", fontweight="bold")
        elif vertex in instance.pickups:
            ax1.plot(position[0], position[1], "o", color="lightgreen", markersize=10)
            ax1.text(position[0], position[1] + 0.3, f"P{vertex}", ha="center")
        else:
            ax1.plot(position[0], position[1], "s", color="lightcoral", markersize=10)
            ax1.text(
                position[0],
                position[1] + 0.3,
                f"D{vertex - instance.num_pairs}",
                ha="center",
            )
    ax1.set_xlabel("Schematic x-coordinate")
    ax1.set_ylabel("Schematic y-coordinate")
    ax1.grid(True, alpha=0.3)
    ax1.legend(title="Route color")

    ax2.set_title("Vehicle Load Profiles", fontweight="bold")
    for index, route_info in enumerate(routes):
        stop_sequence = range(len(route_info["load_profile"]))
        ax2.plot(
            stop_sequence,
            route_info["load_profile"],
            marker="o",
            linewidth=2,
            color=colors[index % len(colors)],
            label=f"Vehicle {route_info['vehicle']}",
        )
    ax2.axhline(
        y=instance.vehicle_capacity,
        color="black",
        linestyle="--",
        alpha=0.7,
        label="Vehicle capacity",
    )
    ax2.set_xlabel("Stop sequence, including depots")
    ax2.set_ylabel("Load (demand units)")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    ax3.set_title("Distance by Vehicle", fontweight="bold")
    vehicle_names = [f"Vehicle {route['vehicle']}" for route in routes]
    route_distances = [route["distance"] for route in routes]
    bars = ax3.bar(
        vehicle_names,
        route_distances,
        color=[colors[index % len(colors)] for index in range(len(routes))],
    )
    ax3.set_ylabel("Travel distance (distance units)")
    ax3.grid(True, axis="y", alpha=0.3)
    for bar, distance in zip(bars, route_distances):
        ax3.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{distance:.1f}",
            ha="center",
            va="bottom",
            fontweight="bold",
        )

    precedence_pairs = 0
    for pickup in instance.pickups:
        delivery = pickup + instance.num_pairs
        matching_route = next(
            route["route"] for route in routes if pickup in route["route"]
        )
        if matching_route.index(pickup) < matching_route.index(delivery):
            precedence_pairs += 1
    precedence_compliance = 100 * precedence_pairs / instance.num_pairs
    capacity_utilization = (
        100
        * sum(route["peak_load"] for route in routes)
        / (len(routes) * instance.vehicle_capacity)
    )
    total_distance = sum(route["distance"] for route in routes)
    metrics = [
        "Total distance",
        "Average distance per vehicle",
        "Fleet peak-load utilization (%)",
        "Precedence compliance (%)",
    ]
    metric_values = [
        total_distance,
        total_distance / len(routes),
        capacity_utilization,
        precedence_compliance,
    ]

    ax4.set_title("Solution Metrics", fontweight="bold")
    metric_bars = ax4.barh(metrics, metric_values, color="steelblue")
    ax4.set_xlabel("Value (metrics use different units)")
    ax4.grid(True, axis="x", alpha=0.3)
    label_offset = max(metric_values) * 0.01 if max(metric_values) > 0 else 0.01
    for bar, value in zip(metric_bars, metric_values):
        ax4.text(
            bar.get_width() + label_offset,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.2f}",
            ha="left",
            va="center",
        )

    plt.tight_layout()
    plt.show()
    return fig


if solution_status == "Optimal":
    fig = visualize_solution(instance, routes)
    print("\nSolution plots generated.")
else:
    print("Visualization is unavailable because optimality was not certified.")

In [ ]:
def sensitivity_analysis(instance: VRPPDInstance):
    """Resolve the same MILP for several vehicle capacities."""
    print("\n=== CAPACITY SENSITIVITY ANALYSIS ===")
    capacities = [4, 6, 8, 10]
    results = []

    for capacity in capacities:
        modified_instance = VRPPDInstance(
            num_pairs=instance.num_pairs,
            num_vehicles=instance.num_vehicles,
            vehicle_capacity=capacity,
            distances=instance.distances.copy(),
            demands=instance.demands.copy(),
        )
        prob_mod, x_mod, u_mod, s_mod = solve_vrppd_mip(modified_instance)
        status = pulp.LpStatus[prob_mod.status]

        if status == "Optimal":
            routes_mod = extract_solution(modified_instance, x_mod, u_mod, s_mod)
            total_distance = sum(route["distance"] for route in routes_mod)
            vehicles_used = sum(
                len(route["route"]) > 2 for route in routes_mod
            )
            results.append({
                "capacity": capacity,
                "total_distance": total_distance,
                "vehicles_used": vehicles_used,
                "status": status,
            })
        else:
            results.append({
                "capacity": capacity,
                "total_distance": None,
                "vehicles_used": None,
                "status": status,
            })

    print("\nCapacity | Total distance | Vehicles used | Solver status")
    print("-" * 66)
    for result in results:
        if result["total_distance"] is None:
            distance_text = "N/A"
            vehicles_text = "N/A"
        else:
            distance_text = f"{result['total_distance']:.2f}"
            vehicles_text = str(result["vehicles_used"])
        print(
            f"{result['capacity']:8d} | {distance_text:14s} | "
            f"{vehicles_text:13s} | {result['status']}"
        )

    return results


sensitivity_results = sensitivity_analysis(instance)

## Interpretation and limitations

A route is an ordered sequence of visits that begins at the start depot and ends at the end depot. Its reported distance is the sum of the corresponding entries in the supplied distance matrix. A load profile is calculated directly from the signed customer demands after each stop; its peak, rather than its ending load, indicates the capacity required by that route. Because each pickup and its paired delivery have equal and opposite demands, a completed route returns with zero physical load.

The load variables in the MILP are auxiliary lower-bound states used to enforce capacity at customer vertices. The displayed load profiles instead track the physical cumulative demand directly, making them suitable for interpreting the route. The extraction checks that each route's physical load remains nonnegative and does not exceed capacity, that all customers are served once, that pairs share a vehicle, and that pickups precede deliveries. It also checks that extracted route distances agree with the MILP objective.

The MILP proves optimality only when CBC reports **Optimal**. The model is exact for the stated assumptions, including the strictly positive customer-to-customer travel durations that allow time transitions to exclude disconnected customer cycles. Its computational requirements can grow quickly as the number of locations increases. The capacity sensitivity analysis keeps distances, demands, and fleet size fixed and solves the same formulation for each listed capacity. A non-optimal solver status is reported as such rather than interpreted as infeasibility or as a proven optimum.

This formulation is useful when a rigorous optimum is important for a small instance, when studying how constraints shape routes, or when creating a benchmark. It assumes static travel costs and a fixed fleet; time windows, service times, and operational changes would require additional modeling constraints.

In [ ]:
def final_summary(instance: VRPPDInstance, prob, routes):
    """Summarize the instance, solver status, and available optimal routes."""
    print("\n" + "=" * 64)
    print("VRPPD MATHEMATICAL FORMULATION — SUMMARY")
    print("=" * 64)

    print("\nProblem characteristics:")
    print(f"- Pickup-delivery pairs: {instance.num_pairs}")
    print(f"- Vehicles available: {instance.num_vehicles}")
    print(f"- Vehicle capacity: {instance.vehicle_capacity}")
    pickup_quantity = sum(
        instance.demands[pickup - 1] for pickup in instance.pickups
    )
    print(f"- Total quantity picked up across all pairs: {pickup_quantity}")

    status = pulp.LpStatus[prob.status]
    print(f"\nSolver status: {status}")
    if status == "Optimal":
        total_distance = sum(route["distance"] for route in routes)
        fleet_peak_load = sum(route["peak_load"] for route in routes)
        fleet_capacity = len(routes) * instance.vehicle_capacity
        capacity_utilization = 100 * fleet_peak_load / fleet_capacity
        vehicles_used = sum(len(route["route"]) > 2 for route in routes)

        print("\nOptimal solution summary:")
        print(f"- Total travel distance: {total_distance:.2f}")
        print(f"- Vehicles serving at least one customer: {vehicles_used}")
        print(f"- Average distance per available vehicle: {total_distance / len(routes):.2f}")
        print(f"- Fleet peak-load utilization: {capacity_utilization:.1f}%")

        print("\nRoute details:")
        for route_info in routes:
            route_labels = []
            for vertex in route_info["route"]:
                if vertex in (instance.depot_start, instance.depot_end):
                    route_labels.append("Depot")
                elif vertex in instance.pickups:
                    route_labels.append(f"P{vertex}")
                else:
                    route_labels.append(f"D{vertex - instance.num_pairs}")
            route_text = " → ".join(route_labels)
            print(f"- Vehicle {route_info['vehicle']}: {route_text}")
            print(
                f"  Distance: {route_info['distance']:.2f}; "
                f"peak load: {route_info['peak_load']:.0f}; "
                f"ending load: {route_info['load']:.0f}"
            )

    print("\nFormulation highlights:")
    print("- Binary routing variables minimize total travel distance.")
    print("- Flow conservation connects each vehicle's route through customers.")
    print("- Pairing and time constraints enforce same-vehicle service and precedence.")
    print("- Load transitions enforce capacity along selected customer arcs.")
    print("- CBC's optimal status is required for an optimality claim.")
    print("=" * 64)


final_summary(instance, prob, routes)